# Step 3 tour: what MLflow actually stores, and where

Three ideas, all of which come back in Steps 4–9:

1. **Metadata in a database, bytes in S3** (same split as Iceberg in notebook 02).
2. **An alias is a movable pointer**, so promotion and rollback are one API call.
3. **The hostname depends on where the code runs** (laptop vs inside Docker).

Plus a bonus: what a memory limit does when it is hit.

Prereq: the stack is running (`docker compose --env-file .env -f infra/docker-compose.yml up -d`).

In [1]:
import os
import subprocess

import mlflow
import pandas as pd
import psycopg
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from murray.io.registry import configure
from murray.io.s3 import delete_prefix, list_keys

client = configure()
EXPERIMENT = "tour"
MODEL = "tour-barmah-height"

# Re-runnable: remove the tour model from a previous run.
if client.search_registered_models(filter_string=f"name = '{MODEL}'"):
    client.delete_registered_model(MODEL)
experiment_id = mlflow.set_experiment(EXPERIMENT).experiment_id
print("tracking URI:", mlflow.get_tracking_uri(), "| experiment id:", experiment_id)

2026/10/24 17:13:57 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/isaacfrith/Murray-MlOps/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


tracking URI: http://localhost:5050 | experiment id: 3


## 1. Log a run and register a model

A toy "model": tomorrow's level = today's level + a fixed offset. Version 1 adds 0.10 m.

In [2]:
class Offset(mlflow.pyfunc.PythonModel):
    def __init__(self, offset: float):
        self.offset = offset

    def predict(self, context, model_input: pd.DataFrame, params=None):
        return model_input["level_m"] + self.offset


def train_and_register(offset: float, mae: float) -> str:
    """Log one run with a model; return the new registered model version."""
    with mlflow.start_run(run_name=f"offset-{offset}") as run:
        mlflow.log_params({"offset_m": offset, "horizon_days": 1})
        for step, value in enumerate([mae * 1.6, mae * 1.2, mae]):  # a metric with history
            mlflow.log_metric("val_mae_m", value, step=step)
        with open("notes.txt", "w") as f:
            f.write(f"offset model, offset={offset}\n")
        mlflow.log_artifact("notes.txt")
        os.remove("notes.txt")
        info = mlflow.pyfunc.log_model(name="model", python_model=Offset(offset),
                                       registered_model_name=MODEL)
    print(f"run {run.info.run_id[:8]}… → {MODEL} v{info.registered_model_version}")
    return info.registered_model_version, run.info.run_id


v1, run1 = train_and_register(offset=0.10, mae=0.080)

2026/10/24 17:13:58 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/10/24 17:13:58 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


2026/10/24 17:13:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Successfully registered model 'tour-barmah-height'.


2026/10/24 17:13:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tour-barmah-height, version 1


Created version '1' of model 'tour-barmah-height'.


🏃 View run offset-0.1 at: http://localhost:5050/#/experiments/3/runs/1b52398b01cc47c7a3150bc20e9f6325
🧪 View experiment at: http://localhost:5050/#/experiments/3
run 1b52398b… → tour-barmah-height v1


## 2. What went into Postgres (metadata)

Connect as the `mlflow` role, read-only queries. MLflow 3 creates ~65 tables (most are for
GenAI tracing); these are the ones that matter for classic tracking + registry.

In [3]:
def sql(query: str, *params) -> pd.DataFrame:
    with psycopg.connect(
        host=os.environ.get("POSTGRES_HOST", "localhost"),
        port=os.environ.get("POSTGRES_PORT", "5432"),
        user="mlflow", password=os.environ["MLFLOW_DB_PASSWORD"], dbname="mlflow",
    ) as conn:
        cur = conn.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


sql("SELECT run_uuid, name, status, artifact_uri FROM runs WHERE run_uuid = %s", run1)

,run_uuid,name,status,artifact_uri
0,1b52398b01cc47c7a3150bc20e9f6325,offset-0.1,FINISHED,mlflow-artifacts:/3/1b52398b01cc47c7a3150bc20e...


In [4]:
sql("SELECT key, value FROM params WHERE run_uuid = %s ORDER BY key", run1)

,key,value
0,horizon_days,1
1,offset_m,0.1


In [5]:
# Every logged metric value is a row (with step + timestamp); latest_metrics caches the last one.
sql("SELECT key, value, step FROM metrics WHERE run_uuid = %s ORDER BY step", run1)

,key,value,step
0,val_mae_m,0.128,0
1,val_mae_m,0.096,1
2,val_mae_m,0.080,2


In [6]:
sql("SELECT name, version, current_stage, source, run_id FROM model_versions WHERE name = %s", MODEL)

,name,version,current_stage,source,run_id
0,tour-barmah-height,1,None,models:/m-511e7a61e5404e0999537b1c5d18f394,1b52398b01cc47c7a3150bc20e9f6325


Note: **no model weights, no files anywhere in Postgres** — only names, numbers and URIs.
`artifact_uri` / `source` are pointers into the artifact store.

## 3. What went into S3 (bytes)

The server (not your laptop) wrote these to `s3://lakehouse/mlflow/<experiment_id>/…`.

In [7]:
for key in list_keys(f"mlflow/{experiment_id}/"):
    if run1 in key or client.get_model_version(MODEL, v1).source.removeprefix("models:/") in key:
        print(key)

mlflow/3/1b52398b01cc47c7a3150bc20e9f6325/artifacts/notes.txt
mlflow/3/models/m-511e7a61e5404e0999537b1c5d18f394/artifacts/MLmodel


mlflow/3/models/m-511e7a61e5404e0999537b1c5d18f394/artifacts/conda.yaml
mlflow/3/models/m-511e7a61e5404e0999537b1c5d18f394/artifacts/python_env.yaml
mlflow/3/models/m-511e7a61e5404e0999537b1c5d18f394/artifacts/python_model.pkl
mlflow/3/models/m-511e7a61e5404e0999537b1c5d18f394/artifacts/requirements.txt


Two groups: the run's own artifacts (`<run_id>/artifacts/notes.txt`) and the model
(`models/m-…/artifacts/`: `MLmodel`, `python_model.pkl`, requirements files). The `MLmodel`
file is what lets *any* MLflow client reload the model later.

## 4. Aliases: promotion and rollback

Inference will always load `models:/<name>@champion`. The training pipeline decides whether a
new version deserves the alias. Here is that rule in miniature.

In [8]:
client.set_registered_model_alias(MODEL, "champion", v1)  # first model: champion by default


def promote_if_better(candidate_version: str) -> bool:
    """Move @champion to the candidate only if its validation MAE is lower."""
    champion = client.get_model_version_by_alias(MODEL, "champion")
    champ_mae = client.get_run(champion.run_id).data.metrics["val_mae_m"]
    cand = client.get_model_version(MODEL, candidate_version)
    cand_mae = client.get_run(cand.run_id).data.metrics["val_mae_m"]
    better = cand_mae < champ_mae
    print(f"champion v{champion.version} MAE={champ_mae:.3f} vs candidate v{candidate_version} "
          f"MAE={cand_mae:.3f} → {'PROMOTE' if better else 'keep champion'}")
    if better:
        client.set_registered_model_alias(MODEL, "champion", candidate_version)
    return better


def champion_prediction() -> float:
    model = mlflow.pyfunc.load_model(f"models:/{MODEL}@champion")
    return float(model.predict(pd.DataFrame({"level_m": [2.71]}))[0])


print("champion predicts:", champion_prediction())

champion predicts: 2.81


In [9]:
v2, _ = train_and_register(offset=0.05, mae=0.060)  # better
promote_if_better(v2)
print("champion predicts:", champion_prediction())

2026/10/24 17:14:02 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/10/24 17:14:02 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


2026/10/24 17:14:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'tour-barmah-height' already exists. Creating a new version of this model...


2026/10/24 17:14:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tour-barmah-height, version 2


Created version '2' of model 'tour-barmah-height'.


🏃 View run offset-0.05 at: http://localhost:5050/#/experiments/3/runs/580aee2393d9404f81d201a0fd5c9df0
🧪 View experiment at: http://localhost:5050/#/experiments/3
run 580aee23… → tour-barmah-height v2


champion v1 MAE=0.080 vs candidate v2 MAE=0.060 → PROMOTE


champion predicts: 2.76


In [10]:
v3, _ = train_and_register(offset=0.30, mae=0.150)  # worse
promote_if_better(v3)
print("champion predicts:", champion_prediction(), "(unchanged: v3 was registered but not promoted)")

2026/10/24 17:14:04 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/10/24 17:14:04 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


2026/10/24 17:14:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'tour-barmah-height' already exists. Creating a new version of this model...
2026/10/24 17:14:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tour-barmah-height, version 3


Created version '3' of model 'tour-barmah-height'.


🏃 View run offset-0.3 at: http://localhost:5050/#/experiments/3/runs/261f29f56fcf4b40962acc900274e5f9
🧪 View experiment at: http://localhost:5050/#/experiments/3
run 261f29f5… → tour-barmah-height v3


champion v2 MAE=0.060 vs candidate v3 MAE=0.150 → keep champion


champion predicts: 2.76 (unchanged: v3 was registered but not promoted)


In [11]:
# The alias lives in Postgres: one row. Promotion = updating it.
sql("SELECT name, alias, version FROM registered_model_aliases WHERE name = %s", MODEL)

,name,alias,version
0,tour-barmah-height,champion,2


In [12]:
# Rollback: a bad champion found in production → point the alias back. No code change, no deploy.
client.set_registered_model_alias(MODEL, "champion", v1)
print("after rollback, champion predicts:", champion_prediction())

after rollback, champion predicts: 2.81


## 5. Lose the bytes, keep the metadata

What if S3 lost the artifacts but Postgres survived? Simulate it on version 3 only.

In [13]:
v3_info = client.get_model_version(MODEL, v3)
# MLflow 3: each logged model has an id (m-…); the version's `source` points at it.
v3_model_id = v3_info.source.removeprefix("models:/")
v3_prefix = f"mlflow/{experiment_id}/models/{v3_model_id}/"
print("deleting", delete_prefix(v3_prefix), "files under", v3_prefix)

# Metadata still fine:
print("v3 still registered:", client.get_model_version(MODEL, v3).status,
      "| its run's MAE:", client.get_run(v3_info.run_id).data.metrics["val_mae_m"])

# But the model itself is gone:
try:
    mlflow.pyfunc.load_model(f"models:/{MODEL}/{v3}")
except Exception as exc:
    print("load failed:", type(exc).__name__, str(exc)[:160])

deleting 5 files under mlflow/3/models/m-a9c41b9834314256b67e7bffa5003686/
v3 still registered: READY | its run's MAE: 0.15


2026/10/24 17:14:07 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at http://localhost:5050/api/2.0/mlflow-artifacts/artifacts/3/models/m-a9c41b9834314256b67e7bffa5003686/artifacts. Returning destination path.


load failed: MlflowException Could not find an "MLmodel" configuration file at "/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/tmp1_fy38o0/MLmodel"


Same lesson as Iceberg: the database says *what exists*, S3 holds *the thing*. Back up both,
or neither is useful. (Version 3 is now a dangling pointer, like an Iceberg table whose files
were deleted.)

## 6. Hostnames: same server, different address

From the laptop, MLflow is `localhost:5050` (the port mapping). Inside the compose network it
is `mlflow:5000` (service name + container port). Test both from both sides.

In [14]:
import socket
import urllib.request


def reachable(url: str) -> str:
    try:
        urllib.request.urlopen(url, timeout=3)
        return "OK"
    except Exception as exc:
        return f"FAIL ({type(exc).__name__})"


print("from the laptop:")
for url in ["http://localhost:5050/health", "http://mlflow:5000/health"]:
    print(f"  {url:35} {reachable(url)}")

from the laptop:
  http://localhost:5050/health        OK
  http://mlflow:5000/health           FAIL (URLError)


In [15]:
CHECK = """
import urllib.request
for url in ["http://localhost:5050/health", "http://mlflow:5000/health", "http://localhost:5000/health"]:
    try:
        urllib.request.urlopen(url, timeout=3); r = "OK"
    except Exception as e:
        r = f"FAIL ({type(e).__name__})"
    print(f"  {url:35} {r}")
"""
# Run the same check from a throwaway container on the compose network (murray_default).
print("from inside Docker (another container on the network):")
print(subprocess.run(
    ["docker", "run", "--rm", "--network", "murray_default", "murray/mlflow:3.17.2", "python", "-c", CHECK],
    capture_output=True, text=True,
).stdout)

from inside Docker (another container on the network):


  http://localhost:5050/health        FAIL (URLError)
  http://mlflow:5000/health           OK
  http://localhost:5000/health        FAIL (URLError)



- `localhost` inside a container means **that container**. Nothing listens on 5050 there.
- `mlflow` resolves only on the compose network, via Docker's DNS.
- So: laptop code uses `MLFLOW_TRACKING_URI=http://localhost:5050`; Airflow tasks (Step 5)
  use `http://mlflow:5000`. Same code, different environment variable.

## 7. Bonus: what a memory limit does

Run a throwaway container with a 64 MB limit and make it allocate 256 MB.

In [16]:
name = "murray-oom-demo"
subprocess.run(["docker", "rm", "-f", name], capture_output=True)
subprocess.run(
    ["docker", "run", "--name", name, "--memory", "64m", "--memory-swap", "64m",
     "murray/mlflow:3.17.2", "python", "-c", "x = bytearray(256 * 1024 * 1024)"],
    capture_output=True,
)
state = subprocess.run(
    ["docker", "inspect", name, "--format", "exit code={{.State.ExitCode}} OOMKilled={{.State.OOMKilled}}"],
    capture_output=True, text=True,
).stdout.strip()
subprocess.run(["docker", "rm", "-f", name], capture_output=True)
print(state)

exit code=137 OOMKilled=true


Exit code **137** = 128 + 9 (SIGKILL): the kernel killed the process; it got no chance to log
an error. With `restart: unless-stopped`, a real service would come straight back, so the
symptom is "keeps restarting / requests fail sometimes". Check with:

```
docker compose --env-file .env -f infra/docker-compose.yml ps        # Restarting?
docker inspect --format '{{.State.OOMKilled}}' murray-mlflow-1        # true?
docker stats --no-stream                                             # near the limit?
```